In [2]:
# Standard libraries
import bisect
import os
import json

# Third-party libraries
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from joblib import dump
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# PyTorch core
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader, TensorDataset

# PyTorch Lightning
import pytorch_lightning as pl
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

# TorchMetrics
from torchmetrics.classification import (
    BinaryAUROC,
    BinaryAveragePrecision,
    BinaryF1Score,
    BinaryPrecision,
    BinaryRecall
)

Import training data

In [1]:
LEAD_TIME = 1
PARTITION = "train"
SHARDS_DIR = f"/gws/nopw/j04/wiser_ewsa/mrakotomanga/EPS/Africa_sharded/processed/t{LEAD_TIME}/{PARTITION}"

In [5]:
class FastVariableShardDataset(Dataset):
    """
    Load sharded (inputs, targets) pairs.
      - inputs  : arbitrary tensor      (e.g. [B, 140, F])
      - targets : binary mask [H, W]    (default 1024×1024)
    Optionally down-sample targets on the fly to output_size.
    """

    def __init__(self, shards_dir: str, sizes_file: str = "sizes.json",
                 output_size: tuple[int, int] = (512, 512)):

        super().__init__()
        self.shard_dir   = shards_dir
        self.output_size = output_size

        # -------- list shard files --------
        self.shard_files = sorted(
            f for f in os.listdir(shards_dir) if f.endswith("_proc.pt")
        )
        if not self.shard_files:
            raise FileNotFoundError("No *_proc.pt files in {}".format(shards_dir))

        # -------- read per-shard sizes ----
        with open(os.path.join(shards_dir, sizes_file)) as fp:
            sizes_dict = json.load(fp)

        self.shard_sizes   = [sizes_dict[f] for f in self.shard_files]
        self.shard_offsets = [0]
        for n in self.shard_sizes[:-1]:
            self.shard_offsets.append(self.shard_offsets[-1] + n)
        self.total_samples = sum(self.shard_sizes)

        # -------- 1-shard cache -----------
        self._cache_path: str | None = None
        self._cache_data: dict | None = None

    # ------------------------------------
    def __len__(self) -> int:
        return self.total_samples

    # ------------------------------------
    def _load_shard(self, fname: str) -> dict:
        """Load a shard with a tiny RAM cache."""
        if fname != self._cache_path:
            self._cache_data = torch.load(
                os.path.join(self.shard_dir, fname), map_location="cpu"
            )
            self._cache_path = fname
        return self._cache_data

    # ------------------------------------
    def __getitem__(self, idx: int):
        if idx < 0 or idx >= self.total_samples:
            raise IndexError(f"Index {idx} out of range 0–{self.total_samples-1}")

        # locate shard by binary search
        shard_idx  = bisect.bisect_right(self.shard_offsets, idx) - 1
        local_idx  = idx - self.shard_offsets[shard_idx]
        shard_data = self._load_shard(self.shard_files[shard_idx])

        x = shard_data["inputs"][local_idx]                # e.g. (140, F)
        y = shard_data["targets"][local_idx].float()       # (H, W) as float32

        # ---- optional down-sampling -----
        if y.ndim == 2:              # (H, W) → (1, H, W) for interpolate
            y = y.unsqueeze(0)
        if self.output_size != y.shape[-2:]:
            y = F.interpolate(y.unsqueeze(0),           # add batch dim
                              size=self.output_size,
                              mode="nearest").squeeze(0)
        y = y.squeeze(0)             # back to (H, W)

        return x, y

In [ ]:
train_ds = FastVariableShardDataset(
    "/work/scratch-nopw2/.../t1/train",
    output_size=(512, 512)        # change to (1024,1024) later if needed
)

loader = torch.utils.data.DataLoader(
    train_ds, batch_size=16, shuffle=True, num_workers=8, pin_memory=True
)

# Model

In [7]:
class ToyStormTransformerNet(nn.Module):
    def __init__(self, input_dim=9, d_model=128, grid_size=512):
        super().__init__()
        self.grid_size = grid_size
        self.d_model   = d_model

        # core → embedding
        self.input_proj = nn.Linear(input_dim, d_model)

        # two-layer transformer encoder
        enc_layer = nn.TransformerEncoderLayer(d_model, nhead=8, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=2)

        # tiny CNN decoder
        self.decoder = nn.Sequential(
            nn.Conv2d(d_model, 64, 3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 32, 3, padding=1),      nn.ReLU(),
            nn.Conv2d(32, 1, 1), nn.Sigmoid()
        )

    def forward(self, x):
        """
        x : (B, 140, 9)  with scaled lat (col 4) and lon (col 5)
        """
        lat  = x[:, :, 4]        # (B, 140)
        lon  = x[:, :, 5]
        mask = x[:, :, 8]        # 1 = real core, 0 = padding

        # encode
        h = self.input_proj(x)   # (B, 140, d_model)
        h = self.encoder(h)      # (B, 140, d_model)

        B, N, _ = h.shape
        grid = torch.zeros(B, self.d_model, self.grid_size, self.grid_size, device=x.device)

        for b in range(B):
            for i in range(N):
                if mask[b, i] == 0:
                    continue
                r = lat[b, i].long().clamp(0, self.grid_size - 1)
                c = lon[b, i].long().clamp(0, self.grid_size - 1)
                grid[b, :, r, c] += h[b, i]

        return self.decoder(grid)   # (B, 1, 512, 512)

# Training

In [8]:
class StormLit(pl.LightningModule):
    def __init__(self, model, lr=1e-4, fss_weight=0.3, k=32):
        super().__init__()
        self.model = model
        self.lr = lr
        self.fss_w = fss_weight
        self.k = k

    # ---------- FSS-SELF ----------------
    @staticmethod
    def fss_loss(p, t, k=32, eps=1e-6):
        p = F.avg_pool2d(p, k, 1, k // 2)
        t = F.avg_pool2d(t, k, 1, k // 2)
        num = ((p - t) ** 2).mean(dim=(1, 2, 3))
        den = (p ** 2 + t ** 2 + eps).mean(dim=(1, 2, 3))
        return 1 - (1 - num / (den + eps)).mean()

    def forward(self, x):
        return self.model(x)

    def _step(self, batch, stage):
        x, y = batch
        y = y.unsqueeze(1).float()           # (B, 1, H, W)
        y_hat = self(x)                      # (B, 1, H, W)
        bce = F.binary_cross_entropy(y_hat, y)
        fss = self.fss_loss(y_hat, y, k=self.k)
        loss = (1 - self.fss_w) * bce + self.fss_w * fss
        self.log(f"{stage}_loss", loss, prog_bar=True)
        return loss

    def training_step(self, batch, batch_idx):
        return self._step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._step(batch, "val")

    def configure_optimizers(self):
        return torch.optim.AdamW(self.parameters(), lr=self.lr)


# FSS Loss

In [ ]:
def compute_fraction(img: torch.Tensor, kernel_size: int) -> torch.Tensor:
    """Compute fraction in local window using average pooling."""
    return F.avg_pool2d(img, kernel_size, stride=1, padding=kernel_size // 2)

def fss_loss(pred: torch.Tensor, target: torch.Tensor, kernel_size: int = 32, eps: float = 1e-6) -> torch.Tensor:
    """
    pred:   (B, 1, H, W) → sigmoid probs (0–1)
    target: (B, 1, H, W) → binary (0 or 1)
    kernel_size: size of spatial window r
    """
    pred_frac = compute_fraction(pred, kernel_size)
    target_frac = compute_fraction(target, kernel_size)

    numerator = ((pred_frac - target_frac) ** 2).mean(dim=(1, 2, 3))
    denominator = (pred_frac ** 2 + target_frac ** 2 + eps).mean(dim=(1, 2, 3))

    fss = 1 - (numerator / (denominator + eps))
    loss = 1 - fss.mean()  # SELF: spatially enhanced loss
    return loss

# Train

In [ ]:
model     = ToyStormTransformerNet(input_dim=9, grid_size=512)
lit_model = StormLit(model)

trainer = pl.Trainer(
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    max_epochs=5,          # bump to 20–50 once it runs
    precision=16,          # AMP on A100
    log_every_n_steps=10
)

trainer.fit(lit_model, ..., ...)


# Eval

In [ ]:
model.eval()
with torch.no_grad():
    x, _ = next(iter(...))
    y_pred = model(x.cuda()).cpu()    # (B,1,512,512)